In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
C   0.000   0.000   1.349
C   0.000   0.000  -1.307
C   0.000   1.180   0.621
C   0.000  -1.180   0.621
N   0.000   1.192  -0.713
N   0.000  -1.192  -0.713
H   0.000   0.000   2.430
H   0.000   0.000  -2.391
H   0.000   2.145   1.115
H   0.000  -2.145   1.115'''
molecule = 'm_C4H4N2'
spin = 0
charge =  0


In [3]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [4]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [5]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

output file: ./m_C4H4N2.txt
orbital  energy   occ
0     -15.562048226301485    2.0
1     -15.562030105606606    2.0
2     -11.274403020804424    2.0
3     -11.25625848741271    2.0
4     -11.256218679423874    2.0
5     -11.218275830095912    2.0
6     -1.274620424864988    2.0
7     -1.1507565191557867    2.0
8     -1.0302948636463023    2.0
9     -0.8435868796062025    2.0
10     -0.836507145020668    2.0
11     -0.6988089012440941    2.0
12     -0.6360441025114281    2.0
13     -0.6131355915110737    2.0
14     -0.5475316217031563    2.0
15     -0.5358716937325512    2.0
16     -0.5342142938923695    2.0
17     -0.40174791945104693    2.0
18     -0.3620083168571866    2.0
19     -0.33853067769083767    2.0
20     -0.3370541799217039    2.0
21     0.21441577042056434    0.0
22     0.22059349635117795    0.0
23     0.45360604926680137    0.0
24     0.5754891200244621    0.0
25     0.639535123571838    0.0
26     0.6564238459063072    0.0
27     0.681134282475441    0.0
28     0.721496

In [6]:
n_elec= 6
n_orb = 6 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([16,19,21,22,23,24])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-262.0405122885287),
 np.float64(-7.061909913646417),
 FCIvector([[-9.14098543e-01,  0.00000000e+00,  1.73174860e-02,
              0.00000000e+00,  9.22914316e-03,  0.00000000e+00,
             -3.70929646e-03,  8.28378095e-02,  0.00000000e+00,
              6.77914438e-03,  1.10918724e-03,  0.00000000e+00,
              3.00414296e-03,  1.11168117e-03,  0.00000000e+00,
             -2.14932295e-02,  0.00000000e+00, -2.21628637e-02,
              0.00000000e+00, -1.41743943e-03],
            [ 0.00000000e+00,  2.92067210e-02,  0.00000000e+00,
              3.43307288e-03,  0.00000000e+00,  1.17200140e-01,
              0.00000000e+00,  0.00000000e+00, -9.03051527e-03,
              0.00000000e+00,  0.00000000e+00,  1.88635842e-03,
              0.00000000e+00,  0.00000000e+00, -9.63361082e-04,
              0.00000000e+00, -9.75703258e-03,  0.00000000e+00,
             -8.47678428e-04,  0.00000000e+00],
            [ 1.73174860e-02,  0.00000000e+00,  1.74381886e-01,
      

In [7]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      -0.914098542507739
[0 1 2]      [0 2 3]      0.017317485971268
[0 1 2]      [0 1 4]      0.009229143155431
[0 1 2]      [1 2 4]      -0.003709296457148
[0 1 2]      [0 3 4]      0.082837809507828
[0 1 2]      [2 3 4]      0.006779144383616
[0 1 2]      [0 1 5]      0.001109187236073
[0 1 2]      [1 2 5]      0.003004142961948
[0 1 2]      [0 3 5]      0.001111681174620
[0 1 2]      [2 3 5]      -0.021493229524429
[0 1 2]      [1 4 5]      -0.022162863746116
[0 1 2]      [3 4 5]      -0.001417439425870
[0 1 3]      [0 1 3]      0.029206721036052
[0 1 3]      [1 2 3]      0.003433072878700
[0 1 3]      [0 2 4]      0.117200140173110
[0 1 3]      [1 3 4]      -0.009030515271858
[0 1 3]      [0 2 5]      0.001886358416536
[0 1 3]      [1 3 5]      -0.000963361081681
[0 1 3]      [0 4 5]      -0.009757032584095
[0 1 3]      [2 4 5]      -0.000847678428285
[0 2 3]      [0 1 2]      0.017317485971268
[0 2 3]      [0 2 3]      0.174381886404112
[0 2 3]      [0 1 4]   